In [2]:
# Step 0 — Imports
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------
# Step 1 — Set up absolute path for offline use
# ---------------------------------------------------------------
# Your EPC CSVs are stored locally on your desktop
data_path = Path("/Users/jakegehrung/Desktop/data_raw/epc_quarterly_csvs")

# Optional: check that folder exists
if not data_path.exists():
    raise FileNotFoundError(f"Folder not found: {data_path}")

# Grab all CSV files (case-insensitive)
csv_files = list(data_path.glob("*.csv")) + list(data_path.glob("*.CSV"))
print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f.name)

# Read and concatenate all CSVs
df_all = pd.concat(
    (pd.read_csv(f, low_memory=False) for f in csv_files),
    ignore_index=True
)

print(f"Total rows (all Scotland): {len(df_all):,}")

Found 40 CSV files:
Q1 2025.csv
Q1 2019.csv
Q1 2018.csv
Q1 2024.csv
Q1 2020.csv
Q4 2016.csv
Q4 2017.csv
Q1 2021.csv
Q1 2023.csv
Q4 2015.csv
Q1 2022.csv
Q3 2016.csv
Q2 2016.csv
Q2 2017.csv
Q3 2017.csv
Q2 2023.csv
Q3 2023.csv
Q3 2022.csv
Q2 2022.csv
Q3 2020.csv
Q2 2020.csv
Q2 2021.csv
Q3 2021.csv
Q3 2025.csv
Q2 2025.csv
Q3 2019.csv
Q2 2019.csv
Q2 2018.csv
Q3 2018.csv
Q2 2024.csv
Q3 2024.csv
Q4 2019.csv
Q4 2018.csv
Q4 2024.csv
Q4 2023.csv
Q4 2022.csv
Q1 2016.csv
Q4 2020.csv
Q4 2021.csv
Q1 2017.csv
Total rows (all Scotland): 1,512,449


In [3]:
# ---------------------------------------------------------------
# Step 2 — Grab all CSVs (case-insensitive)
# ---------------------------------------------------------------
csv_files = list(data_path.glob("*.csv")) + list(data_path.glob("*.CSV"))

print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f.name)


Found 40 CSV files:
Q1 2025.csv
Q1 2019.csv
Q1 2018.csv
Q1 2024.csv
Q1 2020.csv
Q4 2016.csv
Q4 2017.csv
Q1 2021.csv
Q1 2023.csv
Q4 2015.csv
Q1 2022.csv
Q3 2016.csv
Q2 2016.csv
Q2 2017.csv
Q3 2017.csv
Q2 2023.csv
Q3 2023.csv
Q3 2022.csv
Q2 2022.csv
Q3 2020.csv
Q2 2020.csv
Q2 2021.csv
Q3 2021.csv
Q3 2025.csv
Q2 2025.csv
Q3 2019.csv
Q2 2019.csv
Q2 2018.csv
Q3 2018.csv
Q2 2024.csv
Q3 2024.csv
Q4 2019.csv
Q4 2018.csv
Q4 2024.csv
Q4 2023.csv
Q4 2022.csv
Q1 2016.csv
Q4 2020.csv
Q4 2021.csv
Q1 2017.csv


In [5]:
# ---------------------------------------------------------------
# Step 3 — Read and concatenate all CSVs
# ---------------------------------------------------------------
df_all = pd.concat(
    (pd.read_csv(f, low_memory=False) for f in csv_files),
    ignore_index=True
)

print(f"Total rows (all Scotland): {len(df_all):,}")


Total rows (all Scotland): 1,512,449


In [6]:
# ---------------------------------------------------------------
# Step 4 — Inspect all column names
# ---------------------------------------------------------------
print("Columns in dataset:")
print(df_all.columns.tolist())


Columns in dataset:
['BUILDING_REFERENCE_NUMBER', 'OSG_REFERENCE_NUMBER', 'ADDRESS1', 'ADDRESS2', 'ADDRESS3', 'POSTCODE', 'INSPECTION_DATE', 'TYPE_OF_ASSESSMENT', 'LODGEMENT_DATE', 'ENERGY_CONSUMPTION_CURRENT', 'TOTAL_FLOOR_AREA', '3_YR_ENERGY_COST_CURRENT', '3_YR_ENERGY_SAVINGS_POTENTIAL', 'CURRENT_ENERGY_EFFICIENCY', 'CURRENT_ENERGY_RATING', 'POTENTIAL_ENERGY_EFFICIENCY', 'POTENTIAL_ENERGY_RATING', 'ENVIRONMENT_IMPACT_CURRENT', 'CURRENT_ENVIRONMENTAL_RATING', 'ENVIRONMENT_IMPACT_POTENTIAL', 'POTENTIAL_ENVIRONMENTAL_RATING', 'CO2_EMISS_CURR_PER_FLOOR_AREA', 'IMPROVEMENTS', 'WALL_DESCRIPTION', 'WALL_ENERGY_EFF', 'WALL_ENV_EFF', 'ROOF_DESCRIPTION', 'ROOF_ENERGY_EFF', 'ROOF_ENV_EFF', 'FLOOR_DESCRIPTION', 'FLOOR_ENERGY_EFF', 'FLOOR_ENV_EFF', 'WINDOWS_DESCRIPTION', 'WINDOWS_ENERGY_EFF', 'WINDOWS_ENV_EFF', 'MAINHEAT_DESCRIPTION', 'MAINHEAT_ENERGY_EFF', 'MAINHEAT_ENV_EFF', 'MAINHEATCONT_DESCRIPTION', 'MAINHEATC_ENERGY_EFF', 'MAINHEATC_ENV_EFF', 'SECONDHEAT_DESCRIPTION', 'SHEATING_ENERGY_EFF'

In [7]:
# ---------------------------------------------------------------
# Step 5 — Initial inclusive postcode filter
# ---------------------------------------------------------------
# Start broad (FK14 + G63) for analysis
df_fintry_inclusive = df_all[
    df_all["POSTCODE"].str.startswith(("FK14", "G63"), na=False)
].copy()

print(f"Rows after inclusive postcode filter: {len(df_fintry_inclusive)}")

# Optional stricter filter to addresses that explicitly mention Fintry
# df_fintry_strict = df_fintry_inclusive[
#     df_fintry_inclusive["ADDRESS1"].str.contains("Fintry", case=False, na=False) |
#     df_fintry_inclusive["ADDRESS2"].str.contains("Fintry", case=False, na=False)
# ].copy()

Rows after inclusive postcode filter: 3468


In [8]:
# ---------------------------------------------------------------
# Step 6 — Convert inspection date to datetime and sort
# ---------------------------------------------------------------
df_fintry_inclusive["INSPECTION_DATE"] = pd.to_datetime(
    df_fintry_inclusive["INSPECTION_DATE"],
    errors="coerce"
)

df_fintry_inclusive = df_fintry_inclusive.sort_values("INSPECTION_DATE", ascending=False)


In [9]:
# ---------------------------------------------------------------
# Step 7 — Deduplicate using BUILDING_REFERENCE_NUMBER
# ---------------------------------------------------------------
df_fintry_latest = df_fintry_inclusive.drop_duplicates(
    subset="BUILDING_REFERENCE_NUMBER",
    keep="first"
)

print(f"Unique dwellings after deduplication (BUILDING_REFERENCE_NUMBER): {len(df_fintry_latest)}")


Unique dwellings after deduplication (BUILDING_REFERENCE_NUMBER): 3468


In [10]:
# ---------------------------------------------------------------
# Step 8 — Deduplicate using address + postcode as fallback
# ---------------------------------------------------------------
df_fintry_inclusive["dwelling_id"] = (
    df_fintry_inclusive["ADDRESS1"].str.strip().str.upper() + " | " +
    df_fintry_inclusive["POSTCODE"].str.strip().str.upper()
)

df_fintry_inclusive = df_fintry_inclusive.sort_values("INSPECTION_DATE", ascending=False)

df_fintry_latest = df_fintry_inclusive.drop_duplicates(
    subset="dwelling_id",
    keep="first"
)

print(f"Unique dwellings after deduplication (address + postcode): {len(df_fintry_latest)}")


Unique dwellings after deduplication (address + postcode): 3442


In [12]:
# ---------------------------------------------------------------
# Step 9 — Filter strictly to Fintry addresses (case-insensitive)
# ---------------------------------------------------------------
# Combine all address fields for search
df_fintry_only = df_fintry_latest[
    df_fintry_latest[["ADDRESS1", "ADDRESS2", "ADDRESS3"]]
    .fillna("")  # ensure no NaNs break str.contains
    .apply(lambda row: row.str.upper().str.contains("FINTRY").any(), axis=1)
].copy()

print(f"Rows after Fintry-only filter: {len(df_fintry_only)}")

Rows after Fintry-only filter: 136


In [13]:
# ---------------------------------------------------------------
# Step 10 — Save processed Fintry dataset
# ---------------------------------------------------------------

# Define output folder and file name
output_folder = Path("/Users/jakegehrung/Desktop/data_raw/data_processed")
output_file = output_folder / "fintry_epc_master.csv"

# Make sure the folder exists
output_folder.mkdir(parents=True, exist_ok=True)

# Save CSV
df_fintry_only.to_csv(output_file, index=False)

print(f"Saved processed Fintry dataset to: {output_file}")

Saved processed Fintry dataset to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_master.csv


In [14]:
# Check for duplicates
print("Duplicates by dwelling_id:", df_fintry_only.duplicated(subset="dwelling_id").sum())

# Check for missing inspection dates
print("Missing inspection dates:", df_fintry_only["INSPECTION_DATE"].isna().sum())

# Quick stats on EPC ratings
print(df_fintry_only["CURRENT_ENERGY_RATING"].value_counts())

# Optional: summary of floor area and main heating fuel
print(df_fintry_only["TOTAL_FLOOR_AREA"].describe())
print(df_fintry_only["MAIN_FUEL"].value_counts())

Duplicates by dwelling_id: 0
Missing inspection dates: 0
CURRENT_ENERGY_RATING
D    46
E    27
F    21
C    20
B    12
G     6
A     4
Name: count, dtype: int64
count     136
unique     95
top        60
freq        5
Name: TOTAL_FLOOR_AREA, dtype: object
MAIN_FUEL
electricity (not community)                                        62
oil (not community)                                                36
biomass (community)                                                10
LPG (not community)                                                 7
Electricity: electricity, unspecified tariff                        6
dual fuel - mineral + wood                                          2
wood logs                                                           2
Gas: bulk LPG                                                       1
bulk wood pellets                                                   1
bottled LPG                                                         1
mains gas (not community)          

In [16]:
#Parse IMPROVEMENTS, ALTERNATIVE_IMPROVEMENTS, and LZC_ENERGY_SOURCES
import re
import pandas as pd

# Split multiple measures in IMPROVEMENTS into a list of dicts
def parse_improvements(improv_str):
    if pd.isna(improv_str) or improv_str.strip() == "":
        return []
    
    measures = improv_str.split(" | ")
    parsed = []
    for m in measures:
        measure = {}
        # Extract fields using regex
        desc_match = re.search(r"Description:\s*(.+?);", m)
        cost_match = re.search(r"Indicative Cost:\s*¬?£?([\d,]+)\s*-\s*¬?£?([\d,]+);", m)
        saving_match = re.search(r"Typical Saving:\s*(\d+);", m)
        energy_match = re.search(r"Energy Rating after improvement:\s*([A-G]\s*\d+);", m)
        env_match = re.search(r"Environmental Rating after improvement:\s*([A-G]\s*\d+);", m)
        green_match = re.search(r"Green Deal Eligible:\s*([YN])", m)
        
        measure["description"] = desc_match.group(1) if desc_match else None
        if cost_match:
            measure["cost_min"] = int(cost_match.group(1).replace(",", ""))
            measure["cost_max"] = int(cost_match.group(2).replace(",", ""))
        else:
            measure["cost_min"] = measure["cost_max"] = None
        measure["typical_saving"] = int(saving_match.group(1)) if saving_match else None
        measure["energy_rating"] = energy_match.group(1) if energy_match else None
        measure["env_rating"] = env_match.group(1) if env_match else None
        measure["green_deal"] = green_match.group(1) if green_match else None
        
        parsed.append(measure)
    return parsed

df_fintry_only["IMPROVEMENTS_PARSED"] = df_fintry_only["IMPROVEMENTS"].apply(parse_improvements)

# Optional: explode into multiple rows
df_improvements = df_fintry_only.explode("IMPROVEMENTS_PARSED").reset_index(drop=True)
df_improvements = pd.concat([df_improvements.drop(["IMPROVEMENTS_PARSED"], axis=1), 
                             pd.json_normalize(df_improvements["IMPROVEMENTS_PARSED"])], axis=1)


In [18]:
#Parse Element Performance Columns (WALL_*, ROOF_*, etc.)

element_cols = [
    ("WALL_DESCRIPTION", "WALL_ENERGY_EFF", "WALL_ENV_EFF"),
    ("ROOF_DESCRIPTION", "ROOF_ENERGY_EFF", "ROOF_ENV_EFF"),
    ("FLOOR_DESCRIPTION", "FLOOR_ENERGY_EFF", "FLOOR_ENV_EFF"),
    ("WINDOWS_DESCRIPTION", "WINDOWS_ENERGY_EFF", "WINDOWS_ENV_EFF"),
    ("MAINHEAT_DESCRIPTION", "MAINHEAT_ENERGY_EFF", "MAINHEAT_ENV_EFF"),
    ("MAINHEATCONT_DESCRIPTION", "MAINHEATC_ENERGY_EFF", "MAINHEATC_ENV_EFF"),
    ("SECONDHEAT_DESCRIPTION", "SHEATING_ENERGY_EFF", "SHEATING_ENV_EFF"),
    ("HOTWATER_DESCRIPTION", "HOT_WATER_ENERGY_EFF", "HOT_WATER_ENV_EFF"),
    ("LIGHTING_DESCRIPTION", "LIGHTING_ENERGY_EFF", "LIGHTING_ENV_EFF"),
    ("AIR_TIGHTNESS_DESCRIPTION", "AIR_TIGHTNESS_ENERGY_EFF", "AIR_TIGHTNESS_ENV_EFF")
]

def parse_elements(desc_col, energy_col, env_col):
    parsed_list = []
    for desc, energy, env in zip(desc_col, energy_col, env_col):
        if pd.isna(desc):
            parsed_list.append([])
            continue
        # Split multiple elements by '|'
        elements = [d.strip() for d in desc.split("|")]
        element_parsed = []
        for i, e in enumerate(elements):
            element_dict = {"description": e}
            # Match ratings if they exist
            element_dict["energy_rating"] = energy.split("|")[i].strip() if pd.notna(energy) else None
            element_dict["env_rating"] = env.split("|")[i].strip() if pd.notna(env) else None
            element_parsed.append(element_dict)
        parsed_list.append(element_parsed)
    return parsed_list

for desc_col, energy_col, env_col in element_cols:
    new_col = desc_col + "_PARSED"
    df_fintry_only[new_col] = parse_elements(df_fintry_only[desc_col], 
                                             df_fintry_only[energy_col], 
                                             df_fintry_only[env_col])

In [20]:
#Convert Numeric / Cost Columns

numeric_cols = [
    "ENERGY_CONSUMPTION_CURRENT", "TOTAL_FLOOR_AREA", "3_YR_ENERGY_COST_CURRENT",
    "3_YR_ENERGY_SAVINGS_POTENTIAL", "CO2_EMISS_CURR_PER_FLOOR_AREA",
    "CO2_EMISSIONS_CURRENT", "CO2_EMISSIONS_POTENTIAL",
    "HEATING_COST_CURRENT", "HEATING_COST_POTENTIAL",
    "HOT_WATER_COST_CURRENT", "HOT_WATER_COST_POTENTIAL",
    "LIGHTING_COST_CURRENT", "LIGHTING_COST_POTENTIAL",
    "SPACE_HEATING_DEMAND", "WATER_HEATING_DEMAND",
    "IMPACT_LOFT_INSULATION", "IMPACT_CAVITY_WALL_INSULATION", "IMPACT_SOLID_WALL_INSULATION",
    "FLOOR_HEIGHT", "EXTENSION_COUNT", "FIXED_LIGHTING_OUTLETS_COUNT", "LOW_ENERGY_FIXED_LIGHT_COUNT",
    "NUMBER_HABITABLE_ROOMS", "NUMBER_HEATED_ROOMS", "NUMBER_OPEN_FIREPLACES",
    "UNHEATED_CORRIDOR_LENGTH", "WIND_TURBINE_COUNT"
]

for col in numeric_cols:
    df_fintry_only[col] = pd.to_numeric(df_fintry_only[col].astype(str).str.replace(r"[^0-9.]", "", regex=True), errors="coerce")

In [21]:
#Standardize Categorical / Flag Columns

# Uppercase bands
for col in ["CURRENT_ENERGY_RATING", "POTENTIAL_ENERGY_RATING", "CURRENT_ENVIRONMENTAL_RATING", "POTENTIAL_ENVIRONMENTAL_RATING"]:
    df_fintry_only[col] = df_fintry_only[col].str.upper().str.strip()

# Boolean columns
bool_cols = ["MAINS_GAS_FLAG", "SOLAR_WATER_HEATING_FLAG"]
for col in bool_cols:
    df_fintry_only[col] = df_fintry_only[col].map({"Yes": True, "No": False, "N": False, "Y": True})

# Normalize main fuel types
df_fintry_only["MAIN_FUEL"] = df_fintry_only["MAIN_FUEL"].str.lower().str.replace(r"\s*\(.*\)", "", regex=True).str.strip()

In [23]:
#Convert Date Columns

date_cols = ["INSPECTION_DATE", "LODGEMENT_DATE", "CREATED_AT"]
for col in date_cols:
    df_fintry_only["INSPECTION_DATE"] = pd.to_datetime(df_fintry_only["INSPECTION_DATE"], format="%m/%d/%y", errors="coerce")
    df_fintry_only["LODGEMENT_DATE"] = pd.to_datetime(df_fintry_only["LODGEMENT_DATE"], format="%m/%d/%y", errors="coerce")
    df_fintry_only["CREATED_AT"] = pd.to_datetime(df_fintry_only["CREATED_AT"], format="%d/%m/%Y %H:%M:%S", errors="coerce")

In [24]:
#Fix Addendum Text Column

df_fintry_only["ADDENDUM_TEXT"] = df_fintry_only["ADDENDUM_TEXT"].fillna("").str.strip()

In [25]:
# ---------------------------------------------------------------
# Step X — Save fully processed Fintry dataset
# ---------------------------------------------------------------

from pathlib import Path

# Define output folder and file name
output_folder = Path("/Users/jakegehrung/Desktop/data_raw/data_processed")
output_file = output_folder / "fintry_epc_processed.csv"

# Make sure the folder exists
output_folder.mkdir(parents=True, exist_ok=True)

# Save CSV — no index, UTF-8 encoding
df_fintry_only.to_csv(output_file, index=False, encoding="utf-8")

print(f"Saved fully processed Fintry dataset to: {output_file}")

Saved fully processed Fintry dataset to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_processed.csv
